In [ ]:
import importlib.util
if importlib.util.find_spec("spytial") is None:
    try:
        import piplite
        await piplite.install("spytial-diagramming")
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "spytial-diagramming"])
from spytial import *
from spytial.annotations import *

# Disjoint sets (Union-Find): step-by-step

`MAKE-SET`, `FIND-SET`, and `UNION` evolve the disjoint-set forest one
local edit at a time. A `UNION` only rewires a single parent pointer
(plus, occasionally, a rank bump); a `MAKE-SET` adds an isolated node.
No existing node needs to move.

That's the textbook case for `sequence_policy="stability"`: atoms (the
DSU nodes) stay anchored frame-to-frame, and the eye sees just the
single edge that just flipped.

In [ ]:
@attribute(field="key")
@attribute(field="rank")
@hideAtom(selector="int + str")
@orientation(selector='(parent - iden) & (DSUNode->DSUNode)', directions=['above'])
class DSUNode:
    def __init__(self, key):
        self.key = key
        self.parent = self   # MAKE-SET
        self.rank = 0        # tree rank (upper bound on height)


class DisjointSet:
    def __init__(self, seq=None):
        self._seq = seq
        self._nodes = []

    def _snap(self, label):
        if self._seq is not None:
            self._seq.record(
                hideAtom(selector="list")(self._nodes),
                label=label,
            )

    def make_set(self, key):
        node = DSUNode(key)
        self._nodes.append(node)
        self._snap(f"MAKE-SET({key})")
        return node

    def find_set(self, x):
        if x.parent is not x:
            x.parent = self.find_set(x.parent)
        return x.parent

    def union(self, x, y):
        result = self._link(self.find_set(x), self.find_set(y))
        self._snap(f"UNION({x.key}, {y.key})")
        return result

    def _link(self, x_root, y_root):
        if x_root is y_root:
            return x_root
        if x_root.rank > y_root.rank:
            y_root.parent = x_root
            return x_root
        else:
            x_root.parent = y_root
            if x_root.rank == y_root.rank:
                y_root.rank += 1
            return y_root

## Demo

Run a few `MAKE-SET`s, then a chain of `UNION`s. Each call records a frame.

In [ ]:
seq = sequence(
    sequence_policy="stability",
    title="Disjoint sets: MAKE-SET + UNION sequence",
)

ds = DisjointSet(seq=seq)

# MAKE-SET for each element (each call records a frame)
f = ds.make_set("f")
d = ds.make_set("d")
g = ds.make_set("g")
b = ds.make_set("b")
h = ds.make_set("h")
c = ds.make_set("c")
e = ds.make_set("e")

# A sequence of unions; each one records a frame after the link
ds.union(h, b)
ds.union(c, h)
ds.union(c, e)
ds.union(d, g)
ds.union(f, d)
ds.union(f, c)

seq.diagram(method="browser")